# DIM LOCATION

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_crashes.dim_location
USING DELTA
AS

SELECT
  ROW_NUMBER() OVER (ORDER BY lat, lon) AS dim_location_key,
 *
FROM (
    SELECT 
    DISTINCT
    ROUND(LATITUDE, 4) AS lat,
    ROUND(LONGITUDE, 4) AS lon
    FROM workspace.silver_crashes.crashes_table
    WHERE LATITUDE != 0
      AND LONGITUDE !=0
)

# DIM DATES

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_crashes.dim_dates
USING DELTA
AS

SELECT
  ROW_NUMBER() OVER (ORDER BY crash_date) AS dim_date_key,
  crash_date,
  date_part('year' ,crash_date) AS year,
  date_format(crash_date, 'MMMM') AS month
FROM (
    SELECT 
      DISTINCT(CRASH_DATE) as crash_date
      
    FROM workspace.silver_crashes.crashes_table
)

# DIM WEATHER CONDITIONS


In [0]:
%sql
CREATE OR REPLACE TABLE  workspace.gold_crashes.dim_weather_conditions
USING DELTA
AS
SELECT row_number() over(order by weather_con,lighting_con) as dim_condition_key,
*
FROM
(
  SELECT 
  DISTINCT
  WEATHER_CONDITION AS weather_con,
  LIGHTING_CONDITION AS lighting_con
  FROM
  workspace.silver_crashes.crashes_table
)


# CRASH FACT TABLE

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_crashes.fact_crashes
USING DELTA
AS
SELECT
D.dim_date_key,
-- L.dim_location_key,
W.dim_condition_key,
F.damage_cost_estimate,
F.INJURIES_TOTAL as total_injuries,
F.INJURIES_FATAL AS fatal_injuries
FROM 
 workspace.silver_crashes.crashes_table F
LEFT JOIN
 workspace.gold_crashes.dim_location L
ON ROUND(F.LATITUDE,4) = L.lat
AND ROUND(F.LONGITUDE,4) = L.lon
LEFT JOIN
 workspace.gold_crashes.dim_dates D
ON F.CRASH_DATE = D.crash_date
LEFT JOIN
 workspace.gold_crashes.dim_weather_conditions W
  ON F.WEATHER_CONDITION = W.weather_con
AND F.LIGHTING_CONDITION = W.lighting_con
    

# VEHICLES FACT TABLE

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_crashes.fact_vehicles
USING DELTA
AS
SELECT 
row_number() over(order by crash_record_id,vehicle_id) as vehicle_fact_key,
CRASH_RECORD_ID AS crash_record_id,
VEHICLE_ID AS vehicle_id,
MAKE AS vehicle_make,
MODEL AS vehicle_model,
VEHICLE_TYPE AS vehicle_type,
VEHICLE_USE AS vehicle_use,
VEHICLE_DEFECT AS vehicle_defect,
MANEUVER AS manuever,
OCCUPANT_CNT AS total_occupants
FROM 
 workspace.silver_crashes.vehicles_table

# PERSONS FACT TABLE

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_crashes.fact_persons
USING DELTA
AS
SELECT 
row_number() over(order by crash_record_id, vehicle_id, person_id) as person_fact_key,
CRASH_RECORD_ID AS crash_record_id,
PERSON_TYPE AS person_type,
VEHICLE_ID AS vehicle_id,
PERSON_ID AS id,
SEX AS sex,
AGE AS age,
SAFETY_EQUIPMENT AS safety_equipment,
AIRBAG_DEPLOYED AS airbag_deployed,
EJECTION AS ejection,
INJURY_CLASSIFICATION AS injury_classification
FROM 
 workspace.silver_crashes.person_table